# Check predicted EC numbers for unannotated (UP) Yarrowia proteins in the

Twin of `check_mevalonate_predictions_multilayer.py` — identical code, cell-split for interactive use. Shared helpers come from `notebooks/definitions.py`.

In [ ]:
"""
Check predicted EC numbers for unannotated (UP) Yarrowia proteins in the
multilayer cross-translation parquet against Mevalonate pathway enzymes.

Input:
  - data/predictions_and_results/yarrowia_crosstrans_multilayer_predicted.parquet
    (output of evaluate_crosstrans_multilayer.py)

For every UP (unannotated) protein, the top-3 level-4 (full EC) predictions
are matched against the EC numbers of the Mevalonate pathway. Matches are
printed grouped by matched EC.
"""
import os
import csv
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))  # notebooks/
from definitions import RAW_DIR, PRED_DIR
import pandas as pd
import numpy as np
MEVALONATE_CSV = RAW_DIR / "mevalonate_pathway_EC_numbers.csv"
PARQUET_PATH = PRED_DIR / "yarrowia_crosstrans_multilayer_predicted.parquet"

In [ ]:
def main():
    # 1. Load Mevalonate Pathway EC numbers
    print(f"Loading Mevalonate pathway info from: {MEVALONATE_CSV}")
    mevalonate_ecs = {}
    with open(MEVALONATE_CSV, mode="r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            ec = row["EC Number"].strip()
            mevalonate_ecs[ec] = {
                "Step": row["Step"],
                "Enzyme": row["Enzyme"],
                "Genes": row["Common Gene Name(s)"],
                "Reaction": row["Reaction"]
            }
    print(f"  Loaded {len(mevalonate_ecs)} unique EC numbers from Mevalonate pathway:\n" +
          "\n".join([f"    - {ec}: {info['Enzyme']}" for ec, info in mevalonate_ecs.items()]))

    # 2. Analyze the multilayer cross-translation predicted parquet
    print(f"\n{'='*80}")
    print("Analyzing Predictions from: Deep Hierarchical MLP (Back-Translated Multilayer)")
    print(f"Path: {PARQUET_PATH}")
    print(f"{'='*80}")

    if not os.path.exists(PARQUET_PATH):
        print(f"  [Skipped] File does not exist yet. Please run evaluate_crosstrans_multilayer.py first.")
        return

    df = pd.read_parquet(PARQUET_PATH)
    print(f"  Loaded {len(df)} rows")

    # Filter for Uncategorized Proteins (UP)
    up_df = df[df["EC_Numbers"] == "UP"].copy()
    print(f"  Found {len(up_df)} UP (uncategorized) genes")

    matches = []
    for idx, row in up_df.iterrows():
        gene_name = str(row["Gene_Name"]).strip() if "Gene_Name" in row.index and str(row["Gene_Name"]).strip() else f"Row_{idx} (Unknown Gene)"

        # Check all top-3 predictions at Level 4 (EC4)
        for rank in [1, 2, 3]:
            pred_col = f"Predicted_EC_L4_top{rank}"
            prob_col = f"Prob_L4_top{rank}"

            if pred_col not in row:
                continue

            pred_ec = str(row[pred_col]).strip()
            prob = row[prob_col] if (prob_col and prob_col in row) else np.nan

            if pred_ec in mevalonate_ecs:
                matches.append({
                    "Gene_Name": gene_name,
                    "Matched_EC": pred_ec,
                    "Rank": rank,
                    "Prob": prob,
                    "Enzyme": mevalonate_ecs[pred_ec]["Enzyme"],
                    "Step": mevalonate_ecs[pred_ec]["Step"],
                    "Reaction": mevalonate_ecs[pred_ec]["Reaction"],
                    "Common_Genes": mevalonate_ecs[pred_ec]["Genes"]
                })

    if len(matches) == 0:
        print("\n  No mevalonate pathway EC numbers were predicted for UP genes.")
    else:
        print(f"\n  Found {len(matches)} predictions matching Mevalonate pathway enzymes:")
        match_df = pd.DataFrame(matches)
        # Sort by Matched EC and probability descending
        match_df = match_df.sort_values(by=["Matched_EC", "Rank", "Prob"], ascending=[True, True, False])

        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)

        for ec, group in match_df.groupby("Matched_EC"):
            print(f"\n  Matched EC: {ec} ({mevalonate_ecs[ec]['Enzyme']})")
            print(f"  Step {mevalonate_ecs[ec]['Step']} | Reaction: {mevalonate_ecs[ec]['Reaction']}")
            print(f"  Related Reference Genes: {mevalonate_ecs[ec]['Genes']}")
            print("-" * 100)
            print(group[["Gene_Name", "Rank", "Prob"]].to_string(index=False))
            print("-" * 100)

In [ ]:
if __name__ == "__main__":
    main()